# 4. Advanced Behavioral Feature Engineering

## Objective

Transform historical transaction data into customer-level behavioral features.

All features are engineered strictly from pre-cutoff historical data to avoid leakage.

In [6]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.config import PROCESSED_DATA_PATH
from src.features import *

In [7]:
historical_path = os.path.join(PROCESSED_DATA_PATH, "historical_window.csv")
df = pd.read_csv(historical_path)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


## Feature Categories

### RFM Features
- Recency
- Frequency
- Monetary Value

### Behavioral Stability Metrics
- Interpurchase Gap Mean
- Interpurchase Gap Std (volatility)
- Basket Value Mean
- Basket Value Std

### Trend & Engagement Metrics
- Revenue Trend (linear regression slope)
- Product Diversity Ratio
- Seasonality Ratio

---

## Behavioral Rationale

Customers likely to churn often exhibit:
- Increasing inactivity
- Unstable purchasing intervals
- Declining revenue trend
- Low engagement diversity

These engineered features capture such behavioral signals.

In [8]:
rfm = create_rfm(df)
gap = interpurchase_features(df)
basket = basket_features(df)
trend = revenue_trend(df)
diversity = category_diversity(df)
season = seasonality_ratio(df)

In [9]:
features_df = rfm.merge(gap, on="CustomerID", how="left") \
                 .merge(basket, on="CustomerID", how="left") \
                 .merge(trend, on="CustomerID", how="left") \
                 .merge(diversity, on="CustomerID", how="left") \
                 .merge(season, on="CustomerID", how="left")

In [10]:
churn_path = os.path.join(PROCESSED_DATA_PATH, "churn_labels.csv")
churn_df = pd.read_csv(churn_path)

final_df = features_df.merge(churn_df, on="CustomerID", how="inner")

In [11]:
final_df.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "modeling_dataset.csv"),
    index=False
)

In [12]:
final_df.head()

,CustomerID,Recency,Frequency,Monetary,GapMean,GapStd,BasketMean,BasketStd,RevenueTrend,DiversityRatio,SeasonalityRatio,Churn
0,12346.0,225,12,77556.46,11.969697,40.413090,6463.038333,22271.229481,388.591746,2.250000,0.996353,1
1,12347.0,29,6,4114.18,1.402062,8.835468,685.696667,373.823600,0.053552,17.833333,0.346018,0
2,12348.0,148,4,1709.40,4.021277,16.192442,427.350000,317.465363,1.152607,6.250000,0.522289,0
3,12349.0,307,3,2671.14,1.782178,16.200990,890.380000,620.785076,-0.015965,30.000000,0.525102,0
4,12350.0,210,1,334.40,0.000000,0.000000,334.400000,NaN,-0.309559,17.000000,1.000000,1


In [13]:
final_df["Churn"].value_counts(normalize=True)

Churn
1    0.573062
0    0.426938
Name: proportion, dtype: float64

In [14]:
final_df.isnull().sum()

CustomerID             0
Recency                0
Frequency              0
Monetary               0
GapMean              108
GapStd               164
BasketMean             0
BasketStd           1584
RevenueTrend           0
DiversityRatio         0
SeasonalityRatio       0
Churn                  0
dtype: int64

## Output

A final modeling dataset containing:
- Engineered features
- Time-based churn label

This dataset is ready for model training.